In [4]:
import os
import glob
from contextlib import redirect_stdout
from pathlib import Path
from xdem.coreg import NuthKaab, CoregPipeline
import matplotlib.pyplot as plt

import numpy as np
import rasterio
import geopandas as gpd
import pandas as pd
import xdem
from rasterio.features import geometry_mask, shapes
from shapely.geometry import shape

# the coregistration step here is identify into two main steps:
## coregister lidar 2024 to lidar 2021 using manual identify stable area. 
## identify the stable area depending on their accuracy 
## re apply the stable are to coregister all Sensor DEMs to the LiDAR 2024

# =========================================================
# INPUTS
# =========================================================
res="1m"
OUT_NODATA = -9999.0

# =========================================================
# ITERATION CONTROL
# =========================================================
USE_ITERATIVE_STABLE_SELECTION = True
# True  = remove bad polygons after coregistration and repeat
# False = do only one stable-area selection + one coregistration, then stop
MAX_STABLE_SELECTION_ITERATIONS = 3

# Same robust RMSE-outlier strength for all sensors
AUTO_RMSE_OUTLIER_K = 2

# Minimum TOTAL valid pixels required for Nuth & Kääb coregistration
SATELLITE_MIN_COREG_PIXELS = 1000

DRONE_MIN_COREG_PIXELS_BY_RES = {
    "1m": 1000,
    "2m": 700,
    "3m": 500,
}

# =========================================================
base_dir = Path("/mnt/summer/USERS/KOAI/Sensors_Data_DEMs_LaBerarde")

DEMs_Berarde = {
    "LiDAR_2021": base_dir / f"{res}_resolution/Berared_LiDAR_dsm_{res}_Ellps2021.tif",
    "Pleiades_2024": base_dir / f"{res}_resolution/Pleiades_BFM_MGM_ck9_{res}.tif",
}

Shapefiles = {
    "stable_area_shp": base_dir / f"shapefile/Stable_DoD_ByResolution/Stable_DoD_LiDAR_{res}.shp",
    "Watershed_boundary": base_dir / "shapefile" / "Watershed_mod_final.shp",
}

output_corg = base_dir / f"Corg_{res}/test"
output_analysis = base_dir / f"Analysis_{res}/test"

In [5]:
# =========================================================
# HELPERS
# =========================================================

def resolve_dem_path(path_in):
    """
    Resolve DEM path if input is either:
    - a direct .tif file
    - a directory containing .tif files
    """
    path_in = str(path_in)

    if os.path.isfile(path_in):
        return path_in

    if os.path.isdir(path_in):
        tif_candidates = sorted(glob.glob(os.path.join(path_in, "*.tif")))

        if len(tif_candidates) == 0:
            print(f"[WARNING] No .tif found in directory: {path_in}", flush=True)
            return None

        if len(tif_candidates) > 1:
            print(
                f"[INFO] Multiple .tif files found in {path_in}. "
                f"Using first: {tif_candidates[0]}",
                flush=True,
            )

        return tif_candidates[0]

    print(f"[WARNING] Path does not exist: {path_in}", flush=True)
    return None


def find_intersection_error(dem_path):
    """
    Find ASP IntersectionErr raster if it exists beside the DEM.
    Example:
        DEM.tif -> DEM-IntersectionErr.tif
    """
    if dem_path is None:
        return None

    dem_path = str(dem_path)
    base, ext = os.path.splitext(dem_path)
    candidate = f"{base}-IntersectionErr{ext}"

    if os.path.exists(candidate):
        return candidate

    return None


def dem_to_nan_array(dem):
    """
    Convert xdem.DEM or DEM-like object to float array and standardize invalid pixels to NaN.
    """
    arr = dem.data

    if np.ma.isMaskedArray(arr):
        arr = arr.filled(np.nan)

    arr = np.asarray(arr, dtype=np.float32)

    arr[arr == -9999] = np.nan
    arr[arr == -32767] = np.nan
    arr[arr == -32768] = np.nan

    dem_nodata = getattr(dem, "nodata", None)

    if dem_nodata is not None:
        try:
            arr[arr == dem_nodata] = np.nan
        except Exception:
            pass

    arr[~np.isfinite(arr)] = np.nan

    return arr


def sanitize_dem_inplace(dem):
    """
    Force DEM internal data to use a masked array with NaNs masked.
    """
    arr = dem_to_nan_array(dem)
    dem.data = np.ma.masked_invalid(arr)
    return dem


def get_valid_mask(dem):
    """
    Return valid finite-pixel mask from a DEM.
    """
    return np.isfinite(dem_to_nan_array(dem))


def build_mask_from_gdf(gdf, ref_dem):
    """
    Rasterize polygons to the reference DEM grid.
    True = inside polygons.
    """
    if len(gdf) == 0:
        return np.zeros(ref_dem.shape, dtype=bool)

    return geometry_mask(
        geometries=gdf.geometry,
        out_shape=ref_dem.shape,
        transform=ref_dem.transform,
        invert=True,
    )


def mask_to_valid_polygon_gdf(valid_mask, ref_dem):
    """
    Convert a valid-data raster mask to polygon footprint.
    """
    geoms = []

    for geom, value in shapes(valid_mask.astype(np.uint8), transform=ref_dem.transform):
        if value == 1:
            geoms.append(shape(geom))

    if len(geoms) == 0:
        return gpd.GeoDataFrame(geometry=[], crs=ref_dem.crs)

    gdf = gpd.GeoDataFrame(geometry=geoms, crs=ref_dem.crs)
    gdf = gdf[gdf.geometry.notnull() & ~gdf.geometry.is_empty].copy()
    gdf = gdf.explode(index_parts=False).reset_index(drop=True)

    return gdf


def clip_polygons_to_dem_overlap(stable_gdf, ref_dem, target_dem):
    """
    Clip stable polygons to the common valid footprint of reference and target DEMs.
    """
    ref_valid = get_valid_mask(ref_dem)
    tgt_valid = get_valid_mask(target_dem)

    common_valid = ref_valid & tgt_valid

    valid_footprint = mask_to_valid_polygon_gdf(common_valid, ref_dem)

    if len(valid_footprint) == 0:
        return gpd.GeoDataFrame(geometry=[], crs=ref_dem.crs)

    clipped = gpd.overlay(
        stable_gdf,
        valid_footprint,
        how="intersection",
        keep_geom_type=True,
    )

    if len(clipped) == 0:
        return gpd.GeoDataFrame(geometry=[], crs=ref_dem.crs)

    clipped = clipped[clipped.geometry.notnull() & ~clipped.geometry.is_empty].copy()
    clipped = clipped.explode(index_parts=False).reset_index(drop=True)

    return clipped


def save_single_band_dem(output_path, corrected_dem, reference_dem, nodata_value=-9999.0):
    """
    Save corrected DEM using the grid, CRS, and transform of reference_dem.
    """
    arr = corrected_dem

    if np.ma.isMaskedArray(arr):
        arr = arr.filled(np.nan)

    arr = np.asarray(arr, dtype=np.float32).squeeze()
    arr[~np.isfinite(arr)] = np.nan

    arr_out = np.where(np.isfinite(arr), arr, nodata_value).astype(np.float32)

    profile = {
        "driver": "GTiff",
        "height": arr_out.shape[0],
        "width": arr_out.shape[1],
        "count": 1,
        "dtype": "float32",
        "crs": reference_dem.crs,
        "transform": reference_dem.transform,
        "nodata": nodata_value,
        "compress": "lzw",
    }

    with rasterio.open(output_path, "w", **profile) as dst:
        dst.write(arr_out, 1)


def filter_polygons_by_min_pixels_after_dod(
    stable_gdf: gpd.GeoDataFrame,
    dod_array: np.ndarray,
    transform,
    min_pixels: int = 1,
    intersection_error: np.ndarray | None = None,
    intersection_threshold: float = 0.5,
    clip_range: tuple = (-30, 30),
) -> gpd.GeoDataFrame:
    """
    Initial weak filtering only.

    Purpose:
        Remove polygons with no valid pixels or almost no valid pixels.

    This is NOT the main stable-area quality filter.
    Main filtering is later done automatically using polygon-level RMSE.
    """
    keep_rows = []
    dropped_ids = []

    for idx, row in stable_gdf.iterrows():

        poly_id = row.get("stable_uid", row.get("id", idx))

        if row.geometry is None or row.geometry.is_empty:
            dropped_ids.append((poly_id, 0))
            continue

        mask_poly = geometry_mask(
            geometries=[row.geometry],
            out_shape=dod_array.shape,
            transform=transform,
            invert=True,
        )

        valid_mask = mask_poly & np.isfinite(dod_array)

        if intersection_error is not None:
            valid_mask = (
                valid_mask
                & np.isfinite(intersection_error)
                & (intersection_error <= intersection_threshold)
            )

        vals = dod_array[valid_mask]

        vals = vals[
            np.isfinite(vals)
            & (vals >= clip_range[0])
            & (vals <= clip_range[1])
        ]

        if vals.size == 0:
            dropped_ids.append((poly_id, 0))
            continue

        med = np.nanmedian(vals)
        nmad_val = 1.4826 * np.nanmedian(np.abs(vals - med))

        if np.isfinite(nmad_val) and nmad_val > 0:
            vals = vals[np.abs(vals - med) <= 3.0 * nmad_val]

        n_final = vals.size

        if n_final >= min_pixels:
            keep_rows.append(row)
        else:
            dropped_ids.append((poly_id, n_final))

    if dropped_ids:
        print(
            f"[INFO] Dropped {len(dropped_ids)} polygons with < {min_pixels} valid pixels.",
            flush=True,
        )

    if len(keep_rows) == 0:
        return gpd.GeoDataFrame(geometry=[], crs=stable_gdf.crs)

    filtered = gpd.GeoDataFrame(
        keep_rows,
        columns=stable_gdf.columns,
        crs=stable_gdf.crs,
    )

    filtered = filtered.reset_index(drop=True)

    return filtered


def print_enhanced_summary(dem_name, shifts):
    """
    Print XY/Z residual shift estimates before and after Nuth & Kääb.
    """
    print(f"\n=== {dem_name} Coregistration Results ===", flush=True)
    print("             |    BEFORE    |    AFTER", flush=True)
    print("-------------|--------------|--------------", flush=True)
    print(f"dx (m)       | {shifts['dx_before']:>+8.3f} | {shifts['dx_after']:>+8.3f}", flush=True)
    print(f"dy (m)       | {shifts['dy_before']:>+8.3f} | {shifts['dy_after']:>+8.3f}", flush=True)
    print(f"Horiz (m)    | {shifts['horiz_before']:>+8.3f} | {shifts['horiz_after']:>+8.3f}", flush=True)
    print(f"dz (m)       | {shifts['dz_before']:>+8.3f} | {shifts['dz_after']:>+8.3f}", flush=True)
    print("-" * 50, flush=True)


def Nuth_coregister_xyz(ref_dem, target_dem, stable_mask=None, verbose=True):
    """
    Run Nuth & Kääb coregistration using xDEM.

    Returns:
        aligned_dem
        shifts dictionary
    """
    if isinstance(stable_mask, str) and stable_mask.lower() in ["null", "none"]:
        stable_mask = None

    original_nk = NuthKaab()
    original_nk.fit(ref_dem, target_dem, inlier_mask=stable_mask)

    matrix0 = original_nk.to_matrix()
    dx0, dy0, dz0 = matrix0[0, 3], matrix0[1, 3], matrix0[2, 3]

    nk = NuthKaab()
    pipeline = CoregPipeline([nk])
    pipeline.fit(ref_dem, target_dem, inlier_mask=stable_mask)

    aligned_dem = pipeline.apply(target_dem)

    after_nk = NuthKaab()
    after_nk.fit(ref_dem, aligned_dem, inlier_mask=stable_mask)

    matrix1 = after_nk.to_matrix()
    dx1, dy1, dz1 = matrix1[0, 3], matrix1[1, 3], matrix1[2, 3]

    shifts = {
        "dx_before": dx0,
        "dy_before": dy0,
        "dz_before": dz0,
        "dx_after": dx1,
        "dy_after": dy1,
        "dz_after": dz1,
        "horiz_before": np.sqrt(dx0**2 + dy0**2),
        "horiz_after": np.sqrt(dx1**2 + dy1**2),
    }

    if verbose:
        dem_label = getattr(target_dem, "name", "target_dem")
        print_enhanced_summary(dem_label, shifts)

    return aligned_dem, shifts


def nmad(arr: np.ndarray) -> float:
    """
    Normalized Median Absolute Deviation.
    """
    arr = np.asarray(arr, dtype=float)
    arr = arr[np.isfinite(arr)]

    if arr.size == 0:
        return np.nan

    med = np.nanmedian(arr)

    return 1.4826 * np.nanmedian(np.abs(arr - med))

def robust_nmad_1d(values):
    """
    Robust NMAD for a 1D array.
    """
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]

    if values.size == 0:
        return np.nan

    med = np.nanmedian(values)
    return 1.4826 * np.nanmedian(np.abs(values - med))



def stats(arr: np.ndarray) -> dict:
    """
    Compute basic robust and classical statistics.
    """
    arr = np.asarray(arr, dtype=float)
    arr = arr[np.isfinite(arr)]

    if arr.size == 0:
        return dict(
            count=0,
            mean=np.nan,
            median=np.nan,
            std=np.nan,
            nmad=np.nan,
            rmse=np.nan,
            mae=np.nan,
        )

    return dict(
        count=int(arr.size),
        mean=float(np.nanmean(arr)),
        median=float(np.nanmedian(arr)),
        std=float(np.nanstd(arr)),
        nmad=float(nmad(arr)),
        rmse=float(np.sqrt(np.nanmean(arr**2))),
        mae=float(np.nanmean(np.abs(arr))),
    )


def compute_polygon_dod_stats(
    dod_array: np.ndarray,
    stable_area_gdf: gpd.GeoDataFrame,
    transform,
    label: str,
    intersection_error: np.ndarray | None = None,
    intersection_threshold: float = 0.5,
    clip_range: tuple = (-30, 30),

    # Optional filtering
    filter_bad_polygons: bool = False,
    median_abs_threshold: float | None = None,
    rmse_threshold: float | None = None,
    min_count: int | None = None,

    # Optional automatic polygon-RMSE filtering
    auto_rmse_filter: bool = False,
    auto_rmse_k: float = 3.0,

    # Log control
    print_table: bool = True,
    print_rejected: bool = False,
) -> tuple[list[dict], dict]:
    """
    Compute per-polygon DoD statistics.

    Main outputs:
        selected_results:
            list of polygon statistics dictionaries.

        selected_pixel_masks:
            dictionary of final valid-pixel masks per selected polygon.

    Notes:
        - Pixel-level filtering is applied inside each polygon:
            1. finite values
            2. optional intersection-error filtering
            3. absolute DoD clipping
            4. 3 × NMAD pixel-level filtering

        - Polygon-level filtering can be:
            1. no filtering
            2. manual median/RMSE/count filtering
            3. automatic RMSE outlier filtering using:
               threshold = median(RMSE) + k × NMAD(RMSE)
    """

    polygon_results_all = []
    polygon_pixel_masks_all = {}

    stable_reset = stable_area_gdf.reset_index(drop=True)

    # =====================================================
    # 1) Compute statistics for all polygons
    # =====================================================
    for poly_uid, row in stable_reset.iterrows():

        poly_geom = row.geometry

        if poly_geom is None or poly_geom.is_empty:
            continue

        poly_id_original = row.get("id", poly_uid)
        stable_uid = int(row.get("stable_uid", poly_uid))

        mask_poly = geometry_mask(
            geometries=[poly_geom],
            out_shape=dod_array.shape,
            transform=transform,
            invert=True,
        )

        valid_mask_initial = mask_poly & np.isfinite(dod_array)

        if intersection_error is not None:
            valid_mask_initial = (
                valid_mask_initial
                & np.isfinite(intersection_error)
                & (intersection_error <= intersection_threshold)
            )

        if not np.any(valid_mask_initial):
            continue

        flat_indices_initial = np.flatnonzero(valid_mask_initial)
        vals = dod_array.ravel()[flat_indices_initial]

        # -------------------------------------------------
        # DoD clipping
        # -------------------------------------------------
        keep_clip = (
            np.isfinite(vals)
            & (vals >= clip_range[0])
            & (vals <= clip_range[1])
        )

        vals_clip = vals[keep_clip]
        flat_indices_clip = flat_indices_initial[keep_clip]

        if vals_clip.size == 0:
            continue

        # -------------------------------------------------
        # Pixel-level NMAD filtering inside polygon
        # -------------------------------------------------
        med = np.nanmedian(vals_clip)
        nmad_val = 1.4826 * np.nanmedian(np.abs(vals_clip - med))

        if np.isfinite(nmad_val) and nmad_val > 0:
            keep_nmad = np.abs(vals_clip - med) <= 3.0 * nmad_val
        else:
            keep_nmad = np.ones(vals_clip.shape, dtype=bool)

        vals_f = vals_clip[keep_nmad]
        flat_indices_final = flat_indices_clip[keep_nmad]

        if vals_f.size == 0:
            continue

        final_mask = np.zeros(dod_array.shape, dtype=bool)
        final_mask.ravel()[flat_indices_final] = True

        s = stats(vals_f)

        polygon_results_all.append({
            "poly_uid": int(poly_uid),          # index in the current stable_area_gdf
            "stable_uid": int(stable_uid),      # original stable polygon ID
            "poly_id": poly_id_original,
            "values": vals_f,
            **s,
        })

        polygon_pixel_masks_all[int(poly_uid)] = final_mask

    # =====================================================
    # 2) Automatic polygon-RMSE threshold
    # =====================================================
    auto_rmse_threshold = None
    auto_rmse_median = np.nan
    auto_rmse_nmad = np.nan

    if auto_rmse_filter and len(polygon_results_all) > 0:

        rmse_values = np.array(
            [r["rmse"] for r in polygon_results_all if np.isfinite(r["rmse"])],
            dtype=float,
        )

        if rmse_values.size > 0:

            auto_rmse_median = np.nanmedian(rmse_values)
            auto_rmse_nmad = robust_nmad_1d(rmse_values)

            if not np.isfinite(auto_rmse_nmad) or auto_rmse_nmad == 0:
                auto_rmse_nmad = np.nanstd(rmse_values)

            if not np.isfinite(auto_rmse_nmad):
                auto_rmse_nmad = 0.0

            auto_rmse_threshold = auto_rmse_median + auto_rmse_k * auto_rmse_nmad

    # =====================================================
    # 3) Select / reject polygons
    # =====================================================
    selected_results = []
    rejected_results = []

    for r0 in polygon_results_all:

        r = r0.copy()

        keep_polygon = True
        reject_reasons = []

        # -------------------------------------------------
        # Minimum valid pixels per polygon
        # -------------------------------------------------
        if min_count is not None:
            if r["count"] < min_count:
                keep_polygon = False
                reject_reasons.append(f"count<{min_count}")

        # -------------------------------------------------
        # Manual median threshold
        # -------------------------------------------------
        if median_abs_threshold is not None:
            if (not np.isfinite(r["median"])) or (abs(r["median"]) > median_abs_threshold):
                keep_polygon = False
                reject_reasons.append(f"|median|>{median_abs_threshold}")

        # -------------------------------------------------
        # Manual RMSE threshold
        # -------------------------------------------------
        if rmse_threshold is not None:
            if (not np.isfinite(r["rmse"])) or (r["rmse"] > rmse_threshold):
                keep_polygon = False
                reject_reasons.append(f"RMSE>{rmse_threshold}")

        # -------------------------------------------------
        # Automatic polygon-RMSE threshold
        # -------------------------------------------------
        if auto_rmse_filter:
            if auto_rmse_threshold is None or not np.isfinite(auto_rmse_threshold):
                keep_polygon = False
                reject_reasons.append("auto_RMSE_threshold_invalid")
            elif (not np.isfinite(r["rmse"])) or (r["rmse"] > auto_rmse_threshold):
                keep_polygon = False
                reject_reasons.append(f"auto_RMSE>{auto_rmse_threshold:.3f}")

        r["selected"] = keep_polygon
        r["reject_reason"] = "; ".join(reject_reasons)

        if filter_bad_polygons:
            if keep_polygon:
                selected_results.append(r)
            else:
                rejected_results.append(r)
        else:
            selected_results.append(r)

    selected_pixel_masks = {
        int(r["poly_uid"]): polygon_pixel_masks_all[int(r["poly_uid"])]
        for r in selected_results
        if int(r["poly_uid"]) in polygon_pixel_masks_all
    }

    # =====================================================
    # 4) Print per-polygon table
    # =====================================================
    if print_table:

        print(f"\nElevation Change Summary per Polygon for {label}", flush=True)
        print(f"[INFO] Total polygons with valid statistics: {len(polygon_results_all)}", flush=True)

        if filter_bad_polygons:
            print("[INFO] Polygon-quality filtering applied.", flush=True)

            if auto_rmse_filter:
                print("[INFO] Automatic polygon-RMSE filter applied.", flush=True)
                print(f"[INFO] Auto RMSE k: {auto_rmse_k:.2f}", flush=True)
                print(f"[INFO] Polygon RMSE median: {auto_rmse_median:.3f} m", flush=True)
                print(f"[INFO] Polygon RMSE NMAD: {auto_rmse_nmad:.3f} m", flush=True)

                if auto_rmse_threshold is not None:
                    print(f"[INFO] Auto RMSE threshold: {auto_rmse_threshold:.3f} m", flush=True)
                else:
                    print("[WARNING] Auto RMSE threshold could not be computed.", flush=True)

            if median_abs_threshold is not None:
                print(f"[INFO] Manual median threshold: ±{median_abs_threshold:.2f} m", flush=True)

            if rmse_threshold is not None:
                print(f"[INFO] Manual RMSE threshold: {rmse_threshold:.2f} m", flush=True)

            if min_count is not None:
                print(f"[INFO] Minimum pixels per polygon: {min_count}", flush=True)

            print(f"[INFO] Selected polygons: {len(selected_results)}", flush=True)
            print(f"[INFO] Rejected polygons: {len(rejected_results)}", flush=True)

        print(
            "PolyUID | StableUID | PolyID | Count |  Mean  | Median |  Std   |  NMAD  |  RMSE  |  MAE  ",
            flush=True,
        )
        print(
            "-----------------------------------------------------------------------------------------------",
            flush=True,
        )

        for r in selected_results:
            print(
                f"{int(r['poly_uid']):^7} | "
                f"{int(r['stable_uid']):^9} | "
                f"{str(r['poly_id']):^6} | "
                f"{int(r['count']):^5} | "
                f"{r['mean']:+6.2f} | "
                f"{r['median']:+6.2f} | "
                f"{r['std']:+6.2f} | "
                f"{r['nmad']:+6.2f} | "
                f"{r['rmse']:+6.2f} | "
                f"{r['mae']:+6.2f}",
                flush=True,
            )

        if print_rejected and filter_bad_polygons and len(rejected_results) > 0:

            print("\n[INFO] Rejected polygons", flush=True)
            print(
                "PolyUID | StableUID | PolyID | Count | Median |  RMSE  | Reason",
                flush=True,
            )
            print(
                "-----------------------------------------------------------------------",
                flush=True,
            )

            for r in rejected_results:
                print(
                    f"{int(r['poly_uid']):^7} | "
                    f"{int(r['stable_uid']):^9} | "
                    f"{str(r['poly_id']):^6} | "
                    f"{int(r['count']):^5} | "
                    f"{r['median']:+6.2f} | "
                    f"{r['rmse']:+6.2f} | "
                    f"{r['reject_reason']}",
                    flush=True,
                )

    return selected_results, selected_pixel_masks

def select_polygon_results_by_auto_rmse(
    polygon_results,
    stage_label,
    k=3.0,
):
    """
    Automatically select stable polygons using polygon-level RMSE.

    Rule:
        RMSE_threshold = median(RMSE) + k * NMAD(RMSE)

    Important:
        This function does NOT check the minimum total coregistration pixels.
        The main loop checks total pixels after building the spatial mask.
    """

    if len(polygon_results) == 0:
        info = {
            "stage": stage_label,
            "threshold": np.nan,
            "median_rmse": np.nan,
            "nmad_rmse": np.nan,
            "k_used": float(k),
            "n_input": 0,
            "n_selected": 0,
            "n_rejected": 0,
        }

        return [], [], info

    valid_results = []
    invalid_results = []

    for r in polygon_results:
        r = r.copy()

        if np.isfinite(r["rmse"]):
            valid_results.append(r)
        else:
            r["selected"] = False
            r["reject_reason"] = f"{stage_label}_invalid_RMSE"
            invalid_results.append(r)

    if len(valid_results) == 0:
        info = {
            "stage": stage_label,
            "threshold": np.nan,
            "median_rmse": np.nan,
            "nmad_rmse": np.nan,
            "k_used": float(k),
            "n_input": len(polygon_results),
            "n_selected": 0,
            "n_rejected": len(invalid_results),
        }

        return [], invalid_results, info

    rmse_values = np.array([r["rmse"] for r in valid_results], dtype=float)

    median_rmse = np.nanmedian(rmse_values)
    nmad_rmse = robust_nmad_1d(rmse_values)

    if not np.isfinite(nmad_rmse) or nmad_rmse == 0:
        nmad_rmse = np.nanstd(rmse_values)

    if not np.isfinite(nmad_rmse):
        nmad_rmse = 0.0

    threshold = median_rmse + k * nmad_rmse

    selected_results = []
    rejected_results = []

    for r in valid_results:
        r = r.copy()

        if r["rmse"] <= threshold:
            r["selected"] = True
            r["reject_reason"] = ""
            selected_results.append(r)
        else:
            r["selected"] = False
            r["reject_reason"] = f"{stage_label}_RMSE>{threshold:.3f}"
            rejected_results.append(r)

    rejected_results.extend(invalid_results)

    info = {
        "stage": stage_label,
        "threshold": float(threshold),
        "median_rmse": float(median_rmse),
        "nmad_rmse": float(nmad_rmse),
        "k_used": float(k),
        "n_input": len(polygon_results),
        "n_selected": len(selected_results),
        "n_rejected": len(rejected_results),
    }

    return selected_results, rejected_results, info


def stable_gdf_from_selected_polygon_results(
    stable_area_gdf: gpd.GeoDataFrame,
    selected_polygon_results: list[dict],
) -> gpd.GeoDataFrame:
    """
    Build selected stable-area GeoDataFrame from selected polygon results.

    Uses poly_uid, which corresponds to reset_index position in stable_area_gdf.
    """
    if len(selected_polygon_results) == 0:
        return gpd.GeoDataFrame(geometry=[], crs=stable_area_gdf.crs)

    selected_uids = [int(r["poly_uid"]) for r in selected_polygon_results]

    stable_reset = stable_area_gdf.reset_index(drop=True)

    selected_gdf = stable_reset.iloc[selected_uids].copy()
    selected_gdf = selected_gdf.reset_index(drop=True)

    return selected_gdf


def summarize_polygon_results(polygon_results: list[dict]) -> tuple[dict, dict]:
    """
    Return:
        weighted_stats: weighted by polygon valid-pixel count
        all_stats: pooled over all filtered pixels
    """
    empty = dict(
        count=0,
        mean=np.nan,
        median=np.nan,
        std=np.nan,
        nmad=np.nan,
        rmse=np.nan,
        mae=np.nan,
    )

    if len(polygon_results) == 0:
        return empty, empty

    counts = np.array([r["count"] for r in polygon_results], dtype=float)
    means = np.array([r["mean"] for r in polygon_results], dtype=float)
    medians = np.array([r["median"] for r in polygon_results], dtype=float)
    stds = np.array([r["std"] for r in polygon_results], dtype=float)
    nmads = np.array([r["nmad"] for r in polygon_results], dtype=float)
    rmses = np.array([r["rmse"] for r in polygon_results], dtype=float)
    maes = np.array([r["mae"] for r in polygon_results], dtype=float)

    valid_weight = np.isfinite(counts) & (counts > 0)

    if not np.any(valid_weight):
        return empty, empty

    counts = counts[valid_weight]
    means = means[valid_weight]
    medians = medians[valid_weight]
    stds = stds[valid_weight]
    nmads = nmads[valid_weight]
    rmses = rmses[valid_weight]
    maes = maes[valid_weight]

    weighted_stats = {
        "count": int(np.sum(counts)),
        "mean": float(np.average(means, weights=counts)),
        "median": float(np.average(medians, weights=counts)),
        "std": float(np.average(stds, weights=counts)),
        "nmad": float(np.average(nmads, weights=counts)),
        "rmse": float(np.sqrt(np.average(rmses**2, weights=counts))),
        "mae": float(np.average(maes, weights=counts)),
    }

    all_vals = np.concatenate([
        np.asarray(r["values"], dtype=float)
        for r in polygon_results
        if r["count"] > 0
    ])

    all_stats = stats(all_vals)

    return weighted_stats, all_stats


def print_summary_rows(label: str, weighted_stats: dict, all_stats: dict):
    """
    Print global summary statistics.
    """
    print(f"\nGlobal Summary for {label}", flush=True)
    print("Method | Count |  Mean  | Median |  Std   |  NMAD  |  RMSE  |  MAE", flush=True)
    print("---------------------------------------------------------------------", flush=True)

    print(
        f"{'WEIGHT':^6} | {int(weighted_stats['count']):^5} | "
        f"{weighted_stats['mean']:+6.2f} | {weighted_stats['median']:+6.2f} | "
        f"{weighted_stats['std']:+6.2f} | {weighted_stats['nmad']:+6.2f} | "
        f"{weighted_stats['rmse']:+6.2f} | {weighted_stats['mae']:+6.2f}",
        flush=True,
    )

    print(
        f"{'ALL':^6} | {int(all_stats['count']):^5} | "
        f"{all_stats['mean']:+6.2f} | {all_stats['median']:+6.2f} | "
        f"{all_stats['std']:+6.2f} | {all_stats['nmad']:+6.2f} | "
        f"{all_stats['rmse']:+6.2f} | {all_stats['mae']:+6.2f}",
        flush=True,
    )


def get_sensor_from_dem_name(dem_name: str) -> str:
    """
    Return simplified sensor name.
    """
    name = dem_name.lower()

    if "lidar" in name:
        return "LiDAR"

    if "drone" in name:
        return "Drone"

    if "neo" in name:
        return "PleiadesNEO"

    if "pleiades" in name:
        return "Pleiades"

    if "spot" in name:
        return "SPOT"

    return "Unknown"


def get_coreg_sensor_group(dem_name: str) -> str:
    """
    Return sensor group for coregistration pixel threshold.
    """
    name = dem_name.lower()

    if "drone" in name:
        return "Drone"

    return "Satellite"


def get_coreg_filter_params(res: str, dem_name: str) -> int:
    """
    Return minimum TOTAL valid pixels required for Nuth & Kääb coregistration.

    Drone:
        resolution-dependent threshold.

    Satellite:
        fixed threshold for all resolutions.
    """
    sensor_group = get_coreg_sensor_group(dem_name)

    if sensor_group == "Drone":

        if res not in DRONE_MIN_COREG_PIXELS_BY_RES:
            raise KeyError(
                f"Resolution '{res}' not found in DRONE_MIN_COREG_PIXELS_BY_RES. "
                f"Available: {list(DRONE_MIN_COREG_PIXELS_BY_RES.keys())}"
            )

        return DRONE_MIN_COREG_PIXELS_BY_RES[res]

    return SATELLITE_MIN_COREG_PIXELS


def save_dod_density_plot(
    values_before,
    values_after,
    dem_name,
    output_dir,
    xlim=(-5, 5),
    bins=120,
):
    """
    Save DoD density plot before and after coregistration.
    """
    values_before = np.asarray(values_before, dtype=float)
    values_after = np.asarray(values_after, dtype=float)

    values_before = values_before[np.isfinite(values_before)]
    values_after = values_after[np.isfinite(values_after)]

    plt.figure(figsize=(8, 5))

    plt.hist(values_before, bins=bins, density=True, alpha=0.5, label="Before")
    plt.hist(values_after, bins=bins, density=True, alpha=0.5, label="After")

    plt.axvline(0, linestyle="--", linewidth=1)
    plt.xlim(xlim)

    plt.xlabel("Elevation difference (m)")
    plt.ylabel("Density")
    plt.title(f"DoD density: {dem_name}")

    plt.legend()
    plt.tight_layout()

    out_pdf = os.path.join(output_dir, f"{dem_name}_DoD_density_before_after.pdf")

    plt.savefig(out_pdf, dpi=600, bbox_inches="tight")
    plt.close()

    return out_pdf


def polygon_results_to_pixel_rows(
    polygon_results,
    site_name,
    resolution,
    dem_name,
    stage,
):
    """
    Convert selected polygon DoD values to long-format pixel table.
    """
    rows = []
    sensor = get_sensor_from_dem_name(dem_name)

    for r in polygon_results:

        stable_uid = r.get("stable_uid", r.get("poly_uid", r.get("poly_id")))
        vals = np.asarray(r["values"], dtype=float)
        vals = vals[np.isfinite(vals)]

        for v in vals:
            rows.append({
                "Site": site_name,
                "Resolution": resolution,
                "DEM_name": dem_name,
                "Sensor": sensor,
                "Stage": stage,
                "stable_uid": stable_uid,
                "dh": float(v),
            })

    return rows

def xdem_deramp_check_after_nuth(
    ref_dem,
    aligned_dem,
    stable_mask,
    min_improvement_rmse=0.02,
    min_improvement_nmad=0.02,
):
    """
    Apply xDEM first-degree Deramp after Nuth & Kääb and decide
    whether to retain it.

    Deramp(poly_order=1) fits a planar X/Y residual bias:
        residual tilt / ramp correction.

    Decision:
        Keep Deramp only if RMSE or NMAD improves by the chosen threshold.
    """

    # -----------------------------------------------------
    # Residuals after Nuth & Kääb
    # -----------------------------------------------------
    dod_nuth = dem_to_nan_array(aligned_dem - ref_dem)

    valid_mask = (
        stable_mask
        & np.isfinite(dod_nuth)
        & get_valid_mask(ref_dem)
        & get_valid_mask(aligned_dem)
    )

    vals_nuth = dod_nuth[valid_mask]

    if vals_nuth.size < 100:
        raise ValueError(f"Not enough valid pixels for Deramp check: {vals_nuth.size}")

    stats_nuth = stats(vals_nuth)

    # -----------------------------------------------------
    # xDEM Deramp: first-degree polynomial = planar tilt
    # IMPORTANT: use inlier_mask, not mask
    # -----------------------------------------------------
    deramp = xdem.coreg.Deramp(poly_order=1)

    deramp.fit(
        ref_dem,
        aligned_dem,
        inlier_mask=valid_mask,
    )

    deramped_dem = deramp.apply(aligned_dem)
    deramped_dem = sanitize_dem_inplace(deramped_dem)

    # -----------------------------------------------------
    # Residuals after Deramp
    # -----------------------------------------------------
    dod_deramp = dem_to_nan_array(deramped_dem - ref_dem)
    vals_deramp = dod_deramp[valid_mask]

    stats_deramp = stats(vals_deramp)

    # -----------------------------------------------------
    # Improvement
    # -----------------------------------------------------
    rmse_improvement = stats_nuth["rmse"] - stats_deramp["rmse"]
    nmad_improvement = stats_nuth["nmad"] - stats_deramp["nmad"]

    keep_deramp = (
        (rmse_improvement >= min_improvement_rmse)
        or (nmad_improvement >= min_improvement_nmad)
    )

    if keep_deramp:
        final_dem = deramped_dem
    else:
        final_dem = aligned_dem

    deramp_info = {
        "deramp_poly_order": 1,
        "deramp_applied": bool(keep_deramp),
        "deramp_failed": False,
        "deramp_error": "",

        "deramp_count": int(np.count_nonzero(valid_mask)),

        "nuth_mean": stats_nuth["mean"],
        "nuth_median": stats_nuth["median"],
        "nuth_std": stats_nuth["std"],
        "nuth_nmad": stats_nuth["nmad"],
        "nuth_rmse": stats_nuth["rmse"],
        "nuth_mae": stats_nuth["mae"],

        "deramp_mean": stats_deramp["mean"],
        "deramp_median": stats_deramp["median"],
        "deramp_std": stats_deramp["std"],
        "deramp_nmad": stats_deramp["nmad"],
        "deramp_rmse": stats_deramp["rmse"],
        "deramp_mae": stats_deramp["mae"],

        "deramp_rmse_improvement": float(rmse_improvement),
        "deramp_nmad_improvement": float(nmad_improvement),

        "min_improvement_rmse": float(min_improvement_rmse),
        "min_improvement_nmad": float(min_improvement_nmad),
    }

    return final_dem, deramped_dem, deramp_info

In [6]:
# =========================================================
# MAIN
# =========================================================
os.makedirs(output_corg, exist_ok=True)
os.makedirs(output_analysis, exist_ok=True)

site_name = "Berarde"
reference_name = "LiDAR_2024"
reference_lidar = DEMs_Berarde[reference_name]

log_path = os.path.join(output_analysis, f"{site_name}_coreg_log.txt")

with open(log_path, "w") as log_file, redirect_stdout(log_file):

    print(f"\n====================== SITE: {site_name} ======================", flush=True)
    print(f"Reference DEM: {reference_name}", flush=True)
    print(f"Reference path: {reference_lidar}", flush=True)

    shifts_log = []
    pixel_log = []

    all_rejected_stable_uids = set()
    sensor_rejection_log = []

    # ---------------------------------------------------------
    # Load reference DEM
    # ---------------------------------------------------------
    ref_dem = xdem.DEM(reference_lidar)
    ref_dem = sanitize_dem_inplace(ref_dem)

    # ---------------------------------------------------------
    # Load original stable polygons
    # ---------------------------------------------------------
    stable_area = gpd.read_file(Shapefiles["stable_area_shp"]).to_crs(ref_dem.crs)
    stable_area = stable_area.explode(index_parts=False).reset_index(drop=True)

    if "stable_uid" not in stable_area.columns:
        stable_area["stable_uid"] = np.arange(len(stable_area), dtype=int)

    print(f"[INFO] Original stable polygons: {len(stable_area)}", flush=True)

    # ---------------------------------------------------------
    # Optional drone boundary
    # ---------------------------------------------------------
    drone_boundary = None
    drone_boundary_mask = None

    if "Drone_Boundary" in Shapefiles and os.path.exists(Shapefiles["Drone_Boundary"]):
        drone_boundary = gpd.read_file(Shapefiles["Drone_Boundary"]).to_crs(ref_dem.crs)
        drone_boundary = drone_boundary.explode(index_parts=False).reset_index(drop=True)
        drone_boundary_mask = build_mask_from_gdf(drone_boundary, ref_dem)
        print("[INFO] Drone boundary loaded.", flush=True)
    else:
        print("[INFO] Drone boundary not found. Continue without it.", flush=True)

    # ---------------------------------------------------------
    # Loop over target DEMs
    # ---------------------------------------------------------
    for dem_name, dem_path_raw in DEMs_Berarde.items():

        if dem_name == reference_name:
            continue

        print(f"\n---------------------- Processing: {dem_name} ----------------------", flush=True)

        sensor_group = get_coreg_sensor_group(dem_name)

        min_coreg_pixels = get_coreg_filter_params(
            res=res,
            dem_name=dem_name,
        )

        print(f"[INFO] Sensor group: {sensor_group}", flush=True)
        print(f"[INFO] Minimum total coregistration pixels: {min_coreg_pixels}", flush=True)
        print(f"[INFO] Automatic RMSE outlier k: {AUTO_RMSE_OUTLIER_K}", flush=True)
        print(f"[INFO] Iterative stable selection: {USE_ITERATIVE_STABLE_SELECTION}", flush=True)

        # -----------------------------------------------------
        # Resolve target DEM path
        # -----------------------------------------------------
        dem_path = resolve_dem_path(dem_path_raw)

        if dem_path is None:
            print(f"[WARNING] Could not resolve DEM path for {dem_name}. Skipping.", flush=True)
            continue

        print(f"Resolved DEM path: {dem_path}", flush=True)

        # -----------------------------------------------------
        # Load and reproject target DEM
        # -----------------------------------------------------
        try:
            target_dem = xdem.DEM(dem_path)
            target_dem = target_dem.reproject(ref_dem)
            target_dem = sanitize_dem_inplace(target_dem)
        except Exception as e:
            print(f"[ERROR] Failed to load/reproject DEM {dem_name}: {e}", flush=True)
            continue

        # -----------------------------------------------------
        # Load intersection error if available
        # -----------------------------------------------------
        intersection_error_arr = None
        intersection_error_path = find_intersection_error(dem_path)

        if intersection_error_path is not None:
            print(f"[INFO] Intersection error found: {intersection_error_path}", flush=True)

            try:
                intersection_error = xdem.DEM(intersection_error_path)
                intersection_error = intersection_error.reproject(ref_dem)
                intersection_error = sanitize_dem_inplace(intersection_error)
                intersection_error_arr = dem_to_nan_array(intersection_error)

            except Exception as e:
                print(f"[WARNING] Failed to load/reproject intersection error for {dem_name}: {e}", flush=True)
                intersection_error_arr = None
        else:
            print(f"[INFO] No intersection error raster found for {dem_name}", flush=True)

        # -----------------------------------------------------
        # Start from original stable polygons
        # -----------------------------------------------------
        current_stable_area = stable_area.copy()

        # Drone only: clip stable polygons to drone boundary
        if dem_name.lower().startswith("drone") and drone_boundary is not None:
            try:
                current_stable_area = gpd.overlay(
                    current_stable_area,
                    drone_boundary,
                    how="intersection",
                    keep_geom_type=True,
                )

                current_stable_area = (
                    current_stable_area
                    .explode(index_parts=False)
                    .reset_index(drop=True)
                )

                print("[INFO] Clipped stable polygons by drone boundary.", flush=True)

            except Exception as e:
                print(f"[WARNING] Failed to clip stable polygons by drone boundary: {e}", flush=True)

        # -----------------------------------------------------
        # Clip stable polygons to DEM overlap
        # -----------------------------------------------------
        try:
            current_stable_area = clip_polygons_to_dem_overlap(
                stable_gdf=current_stable_area,
                ref_dem=ref_dem,
                target_dem=target_dem,
            )
        except Exception as e:
            print(f"[ERROR] Failed to clip stable polygons to raster overlap for {dem_name}: {e}", flush=True)
            continue

        if len(current_stable_area) == 0:
            print(f"[WARNING] No stable polygon remains inside raster overlap for {dem_name}. Skipping.", flush=True)
            continue

        print(f"[INFO] Candidate stable polygons after overlap clipping: {len(current_stable_area)}", flush=True)

        # -----------------------------------------------------
        # Raw DoD before coregistration
        # -----------------------------------------------------
        dod_raw = dem_to_nan_array(target_dem - ref_dem)

        # -----------------------------------------------------
        # Initial weak pixel-count filtering
        # -----------------------------------------------------
        current_stable_area = filter_polygons_by_min_pixels_after_dod(
            stable_gdf=current_stable_area,
            dod_array=dod_raw,
            transform=ref_dem.transform,
            min_pixels=1,
            intersection_error=intersection_error_arr,
            intersection_threshold=0.5,
            clip_range=(-30, 30),
        )

        if len(current_stable_area) == 0:
            print(f"[WARNING] No polygon remains after valid-pixel filtering for {dem_name}. Skipping.", flush=True)
            continue

        print(f"[INFO] Candidate stable polygons after valid-pixel filtering: {len(current_stable_area)}", flush=True)

        initial_candidate_uids = set(current_stable_area["stable_uid"].astype(int).tolist())

        # =====================================================
        # AUTOMATIC STABLE-AREA SELECTION
        # =====================================================
        aligned_target_dem = None
        shifts = None
        n_coreg_pixels = 0

        converged = False
        stopped_by_min_pixels = False

        max_iter = MAX_STABLE_SELECTION_ITERATIONS if USE_ITERATIVE_STABLE_SELECTION else 1

        for stable_iter in range(1, max_iter + 1):

            if USE_ITERATIVE_STABLE_SELECTION:
                print(
                    f"\n[ITERATION {stable_iter}] Automatic stable-area selection for {dem_name}",
                    flush=True,
                )
            else:
                print(
                    f"\n[SINGLE PASS] Automatic stable-area selection for {dem_name}",
                    flush=True,
                )

            # -------------------------------------------------
            # BEFORE automatic polygon-RMSE filtering
            # -------------------------------------------------
            try:
                before_selected_results, _ = compute_polygon_dod_stats(
                    dod_array=dod_raw,
                    stable_area_gdf=current_stable_area,
                    transform=ref_dem.transform,
                    label=f"{dem_name}_ITER{stable_iter}_BEFORE_AUTO_SELECTION",
                    intersection_error=intersection_error_arr,
                    intersection_threshold=0.5,
                    clip_range=(-30, 30),

                    filter_bad_polygons=True,
                    auto_rmse_filter=True,
                    auto_rmse_k=AUTO_RMSE_OUTLIER_K,

                    median_abs_threshold=None,
                    rmse_threshold=None,
                    min_count=None,

                    print_table=False,
                    print_rejected=False,
                )

            except Exception as e:
                print(f"[ERROR] BEFORE statistics failed for {dem_name}: {e}", flush=True)
                break

            if len(before_selected_results) == 0:
                print(f"[WARNING] No polygon remains after BEFORE automatic RMSE filtering for {dem_name}. Skipping.", flush=True)
                break

            current_stable_area = stable_gdf_from_selected_polygon_results(
                stable_area_gdf=current_stable_area,
                selected_polygon_results=before_selected_results,
            )

            # -------------------------------------------------
            # Build coregistration mask
            # -------------------------------------------------
            current_stable_mask = build_mask_from_gdf(current_stable_area, ref_dem)

            ref_valid_mask = get_valid_mask(ref_dem)
            tgt_valid_mask = get_valid_mask(target_dem)

            coreg_mask = current_stable_mask & ref_valid_mask & tgt_valid_mask

            if dem_name.lower().startswith("drone") and drone_boundary_mask is not None:
                coreg_mask = coreg_mask & drone_boundary_mask

            if intersection_error_arr is not None:
                coreg_mask = (
                    coreg_mask
                    & np.isfinite(intersection_error_arr)
                    & (intersection_error_arr <= 0.5)
                )

            n_coreg_pixels = int(np.count_nonzero(coreg_mask))

            print(f"[INFO] Selected polygons before coregistration: {len(current_stable_area)}", flush=True)
            print(f"[INFO] Valid pixels used for coregistration: {n_coreg_pixels}", flush=True)

            if n_coreg_pixels < min_coreg_pixels:
                print(
                    f"[WARNING] Coregistration stopped for {dem_name}: "
                    f"valid pixels ({n_coreg_pixels}) < minimum required ({min_coreg_pixels}).",
                    flush=True,
                )
                stopped_by_min_pixels = True
                break

            # -------------------------------------------------
            # Nuth & Kääb coregistration
            # -------------------------------------------------
            try:
                aligned_target_dem, shifts = Nuth_coregister_xyz(
                    ref_dem=ref_dem,
                    target_dem=target_dem,
                    stable_mask=coreg_mask,
                    verbose=False,
                )

                aligned_target_dem = sanitize_dem_inplace(aligned_target_dem)

            except Exception as e:
                print(f"[ERROR] XYZ coregistration failed for {dem_name}: {e}", flush=True)
                break

            # -------------------------------------------------
            # AFTER automatic polygon-RMSE check
            # -------------------------------------------------
            dod_coreg = dem_to_nan_array(aligned_target_dem - ref_dem)

            try:
                after_selected_results, after_pixel_masks = compute_polygon_dod_stats(
                    dod_array=dod_coreg,
                    stable_area_gdf=current_stable_area,
                    transform=ref_dem.transform,
                    label=f"{dem_name}_ITER{stable_iter}_AFTER_AUTO_CHECK",
                    intersection_error=intersection_error_arr,
                    intersection_threshold=0.5,
                    clip_range=(-30, 30),

                    filter_bad_polygons=True,
                    auto_rmse_filter=True,
                    auto_rmse_k=AUTO_RMSE_OUTLIER_K,

                    median_abs_threshold=None,
                    rmse_threshold=None,
                    min_count=None,

                    print_table=False,
                    print_rejected=False,
                )

            except Exception as e:
                print(f"[ERROR] AFTER statistics failed for {dem_name}: {e}", flush=True)
                break

            if len(after_selected_results) == 0:
                print(f"[WARNING] No polygon remains after AFTER automatic RMSE check for {dem_name}.", flush=True)
                break

            after_selected_local_uids = set([int(r["poly_uid"]) for r in after_selected_results])
            all_local_uids = set(range(len(current_stable_area)))
            bad_after_local_uids = sorted(list(all_local_uids - after_selected_local_uids))

            print(f"[INFO] AFTER rejected polygons in this pass: {len(bad_after_local_uids)}", flush=True)

            if not USE_ITERATIVE_STABLE_SELECTION:
                converged = True
                print(f"[INFO] Single-pass stable-area selection accepted for {dem_name}.", flush=True)
                break

            if len(bad_after_local_uids) == 0:
                converged = True
                print(f"[INFO] Stable-area selection converged for {dem_name} at iteration {stable_iter}.", flush=True)
                break

            stable_reset = current_stable_area.reset_index(drop=True)

            bad_original_stable_uids = (
                stable_reset
                .iloc[bad_after_local_uids]["stable_uid"]
                .astype(int)
                .tolist()
            )

            trial_stable_area = (
                stable_reset
                .drop(index=bad_after_local_uids)
                .reset_index(drop=True)
            )

            if len(trial_stable_area) == 0:
                print(f"[WARNING] All polygons would be removed for {dem_name}. Keeping previous valid result.", flush=True)
                converged = True
                break

            trial_mask = build_mask_from_gdf(trial_stable_area, ref_dem)
            trial_coreg_mask = trial_mask & ref_valid_mask & tgt_valid_mask

            if dem_name.lower().startswith("drone") and drone_boundary_mask is not None:
                trial_coreg_mask = trial_coreg_mask & drone_boundary_mask

            if intersection_error_arr is not None:
                trial_coreg_mask = (
                    trial_coreg_mask
                    & np.isfinite(intersection_error_arr)
                    & (intersection_error_arr <= 0.5)
                )

            trial_n_coreg_pixels = int(np.count_nonzero(trial_coreg_mask))

            print(f"[INFO] Trial pixels after removing bad polygons: {trial_n_coreg_pixels}", flush=True)

            if trial_n_coreg_pixels < min_coreg_pixels:
                print(
                    f"[WARNING] Iteration stopped for {dem_name}: removing bad polygons would leave "
                    f"{trial_n_coreg_pixels} pixels, below the minimum {min_coreg_pixels}. "
                    f"Keeping previous valid stable area.",
                    flush=True,
                )

                converged = True
                stopped_by_min_pixels = True
                break

            print(f"[INFO] Removing {len(bad_original_stable_uids)} polygons and repeating coregistration.", flush=True)

            for uid in bad_original_stable_uids:
                sensor_rejection_log.append({
                    "Site": site_name,
                    "Resolution": res,
                    "DEM_name": dem_name,
                    "Sensor_group": sensor_group,
                    "stable_uid": int(uid),
                    "reason": f"Rejected by AFTER automatic RMSE filtering at iteration {stable_iter}",
                })

            all_rejected_stable_uids.update(bad_original_stable_uids)
            current_stable_area = trial_stable_area.copy()

        # =====================================================
        # If no valid coregistration result was produced
        # =====================================================
        if aligned_target_dem is None or shifts is None:
            print(f"[WARNING] No valid coregistration result for {dem_name}. Skipping.", flush=True)
            continue

        # =====================================================
        # FINAL COREGISTRATION USING FINAL STABLE AREA
        # =====================================================
        print(f"\n[FINAL] Re-running coregistration using final selected stable area for {dem_name}", flush=True)

        final_stable_mask = build_mask_from_gdf(current_stable_area, ref_dem)

        ref_valid_mask = get_valid_mask(ref_dem)
        tgt_valid_mask = get_valid_mask(target_dem)

        final_coreg_mask = final_stable_mask & ref_valid_mask & tgt_valid_mask

        if dem_name.lower().startswith("drone") and drone_boundary_mask is not None:
            final_coreg_mask = final_coreg_mask & drone_boundary_mask

        if intersection_error_arr is not None:
            final_coreg_mask = (
                final_coreg_mask
                & np.isfinite(intersection_error_arr)
                & (intersection_error_arr <= 0.5)
            )

        n_coreg_pixels = int(np.count_nonzero(final_coreg_mask))

        print(f"[FINAL] Final selected stable polygons: {len(current_stable_area)}", flush=True)
        print(f"[FINAL] Final valid pixels used for coregistration: {n_coreg_pixels}", flush=True)

        if n_coreg_pixels < min_coreg_pixels:
            print(
                f"[WARNING] Final coregistration skipped for {dem_name}: "
                f"valid pixels ({n_coreg_pixels}) < minimum required ({min_coreg_pixels}).",
                flush=True,
            )
            continue

        try:
            aligned_target_dem, shifts = Nuth_coregister_xyz(
                ref_dem=ref_dem,
                target_dem=target_dem,
                stable_mask=final_coreg_mask,
                verbose=True,
            )

            aligned_target_dem = sanitize_dem_inplace(aligned_target_dem)

        except Exception as e:
            print(f"[ERROR] Final XYZ coregistration failed for {dem_name}: {e}", flush=True)
            continue

        # =====================================================
        # FINAL BEFORE table using final stable polygons
        # =====================================================
        try:
            final_before_results, _ = compute_polygon_dod_stats(
                dod_array=dod_raw,
                stable_area_gdf=current_stable_area,
                transform=ref_dem.transform,
                label=f"{dem_name}_FINAL_BEFORE_SELECTED_STABLE",
                intersection_error=intersection_error_arr,
                intersection_threshold=0.5,
                clip_range=(-30, 30),

                filter_bad_polygons=False,
                auto_rmse_filter=False,

                print_table=True,
                print_rejected=False,
            )

            weighted_before, all_before = summarize_polygon_results(final_before_results)
            print_summary_rows(f"{dem_name}_FINAL_BEFORE_SELECTED_STABLE", weighted_before, all_before)

        except Exception as e:
            print(f"[ERROR] Final BEFORE statistics failed for {dem_name}: {e}", flush=True)
            continue

        # =====================================================
        # FINAL AFTER NUTH table using final stable polygons
        # =====================================================
        try:
            dod_coreg_nuth = dem_to_nan_array(aligned_target_dem - ref_dem)

            final_after_nuth_results, _ = compute_polygon_dod_stats(
                dod_array=dod_coreg_nuth,
                stable_area_gdf=current_stable_area,
                transform=ref_dem.transform,
                label=f"{dem_name}_FINAL_AFTER_NUTH_SELECTED_STABLE",
                intersection_error=intersection_error_arr,
                intersection_threshold=0.5,
                clip_range=(-30, 30),

                filter_bad_polygons=False,
                auto_rmse_filter=False,

                print_table=True,
                print_rejected=False,
            )

            weighted_after_nuth, all_after_nuth = summarize_polygon_results(final_after_nuth_results)
            print_summary_rows(f"{dem_name}_FINAL_AFTER_NUTH_SELECTED_STABLE", weighted_after_nuth, all_after_nuth)

        except Exception as e:
            print(f"[ERROR] Final AFTER NUTH statistics failed for {dem_name}: {e}", flush=True)
            continue

        # =====================================================
        # DERAMP CHECK AFTER NUTH
        # =====================================================
        deramp_failed = False
        deramp_error = ""
        deramp_info = {}

        final_dem_to_save = aligned_target_dem
        final_after_results = final_after_nuth_results
        weighted_after = weighted_after_nuth
        all_after = all_after_nuth
        final_stage_label = "After_Nuth"

        try:
            final_dem_candidate, deramped_dem, deramp_info = xdem_deramp_check_after_nuth(
                ref_dem=ref_dem,
                aligned_dem=aligned_target_dem,
                stable_mask=final_coreg_mask,
                min_improvement_rmse=0.02,
                min_improvement_nmad=0.02,
            )

            # -------------------------------------------------
            # Statistics after Deramp test
            # -------------------------------------------------
            dod_deramp = dem_to_nan_array(deramped_dem - ref_dem)

            final_after_deramp_results, _ = compute_polygon_dod_stats(
                dod_array=dod_deramp,
                stable_area_gdf=current_stable_area,
                transform=ref_dem.transform,
                label=f"{dem_name}_FINAL_AFTER_DERAMP_TEST_SELECTED_STABLE",
                intersection_error=intersection_error_arr,
                intersection_threshold=0.5,
                clip_range=(-30, 30),

                filter_bad_polygons=False,
                auto_rmse_filter=False,

                print_table=True,
                print_rejected=False,
            )

            weighted_after_deramp, all_after_deramp = summarize_polygon_results(final_after_deramp_results)
            print_summary_rows(f"{dem_name}_FINAL_AFTER_DERAMP_TEST_SELECTED_STABLE", weighted_after_deramp, all_after_deramp)

            print("\n[DERAMP COMPARISON]", flush=True)
            print("Method      | Count |  Mean  | Median |  Std   |  NMAD  |  RMSE  |  MAE", flush=True)
            print("--------------------------------------------------------------------------", flush=True)
            print(
                f"{'Nuth':^11} | {int(all_after_nuth['count']):^5} | "
                f"{all_after_nuth['mean']:+6.3f} | {all_after_nuth['median']:+6.3f} | "
                f"{all_after_nuth['std']:+6.3f} | {all_after_nuth['nmad']:+6.3f} | "
                f"{all_after_nuth['rmse']:+6.3f} | {all_after_nuth['mae']:+6.3f}",
                flush=True,
            )
            print(
                f"{'Deramp':^11} | {int(all_after_deramp['count']):^5} | "
                f"{all_after_deramp['mean']:+6.3f} | {all_after_deramp['median']:+6.3f} | "
                f"{all_after_deramp['std']:+6.3f} | {all_after_deramp['nmad']:+6.3f} | "
                f"{all_after_deramp['rmse']:+6.3f} | {all_after_deramp['mae']:+6.3f}",
                flush=True,
            )
            print(
                f"[DERAMP] RMSE improvement: {deramp_info['deramp_rmse_improvement']:+.4f} m",
                flush=True,
            )
            print(
                f"[DERAMP] NMAD improvement: {deramp_info['deramp_nmad_improvement']:+.4f} m",
                flush=True,
            )
            print(
                f"[DERAMP] Applied: {deramp_info['deramp_applied']}",
                flush=True,
            )

            # -------------------------------------------------
            # If Deramp improved enough, use Deramp as final
            # Otherwise keep Nuth as final
            # -------------------------------------------------
            if deramp_info["deramp_applied"]:
                final_dem_to_save = deramped_dem
                final_after_results = final_after_deramp_results
                weighted_after = weighted_after_deramp
                all_after = all_after_deramp
                final_stage_label = "After_Nuth_Deramp"
            else:
                final_dem_to_save = aligned_target_dem
                final_after_results = final_after_nuth_results
                weighted_after = weighted_after_nuth
                all_after = all_after_nuth
                final_stage_label = "After_Nuth"

        except Exception as e:
            deramp_failed = True
            deramp_error = str(e)

            deramp_info = {
                "deramp_poly_order": 1,
                "deramp_applied": False,
                "deramp_failed": True,
                "deramp_error": deramp_error,
            }

            final_dem_to_save = aligned_target_dem
            final_after_results = final_after_nuth_results
            weighted_after = weighted_after_nuth
            all_after = all_after_nuth
            final_stage_label = "After_Nuth"

            print(f"[WARNING] xDEM Deramp check failed for {dem_name}: {deramp_error}", flush=True)

        # -----------------------------------------------------
        # Track final rejected polygons
        # -----------------------------------------------------
        final_selected_uids = set(current_stable_area["stable_uid"].astype(int).tolist())
        rejected_uids_this_sensor = initial_candidate_uids - final_selected_uids

        all_rejected_stable_uids.update(rejected_uids_this_sensor)

        for uid in sorted(rejected_uids_this_sensor):
            sensor_rejection_log.append({
                "Site": site_name,
                "Resolution": res,
                "DEM_name": dem_name,
                "Sensor_group": sensor_group,
                "stable_uid": int(uid),
                "reason": "Rejected by automatic stable-area RMSE filtering",
            })

        print(f"[INFO] Final selected stable polygons for {dem_name}: {len(current_stable_area)}", flush=True)
        print(f"[INFO] Final rejected stable polygons for {dem_name}: {len(rejected_uids_this_sensor)}", flush=True)
        print(f"[INFO] Final accepted stage for {dem_name}: {final_stage_label}", flush=True)

        # -----------------------------------------------------
        # Pixel-level logs
        # -----------------------------------------------------
        pixel_log.extend(
            polygon_results_to_pixel_rows(
                polygon_results=final_before_results,
                site_name=site_name,
                resolution=res,
                dem_name=dem_name,
                stage="Before",
            )
        )

        pixel_log.extend(
            polygon_results_to_pixel_rows(
                polygon_results=final_after_nuth_results,
                site_name=site_name,
                resolution=res,
                dem_name=dem_name,
                stage="After_Nuth",
            )
        )

        if (not deramp_failed) and ("deramp_applied" in deramp_info):
            try:
                pixel_log.extend(
                    polygon_results_to_pixel_rows(
                        polygon_results=final_after_deramp_results,
                        site_name=site_name,
                        resolution=res,
                        dem_name=dem_name,
                        stage="After_Deramp_Test",
                    )
                )
            except Exception:
                pass

        # -----------------------------------------------------
        # Save density plot: before vs final accepted result
        # -----------------------------------------------------
        try:
            all_vals_before = np.concatenate([
                r["values"] for r in final_before_results if r["count"] > 0
            ])

            all_vals_after = np.concatenate([
                r["values"] for r in final_after_results if r["count"] > 0
            ])

            plot_path = save_dod_density_plot(
                values_before=all_vals_before,
                values_after=all_vals_after,
                dem_name=dem_name,
                output_dir=output_analysis,
                xlim=(-5, 5),
                bins=120,
            )

            print(f"[INFO] Saved density plot: {plot_path}", flush=True)

        except Exception as e:
            print(f"[WARNING] Density plot failed for {dem_name}: {e}", flush=True)

        # -----------------------------------------------------
        # Save final accepted co-registered DEM
        # -----------------------------------------------------
        output_name = f"{dem_name}_Coreg_res{res}.tif"
        output_path = os.path.join(output_corg, output_name)

        try:
            save_single_band_dem(
                output_path=output_path,
                corrected_dem=final_dem_to_save.data,
                reference_dem=ref_dem,
                nodata_value=OUT_NODATA,
            )

            print(f"[INFO] Saved final accepted raster: {output_path}", flush=True)

        except Exception as e:
            print(f"[ERROR] save_single_band_dem failed for {dem_name}: {e}", flush=True)

        # -----------------------------------------------------
        # Save summary row
        # -----------------------------------------------------
        summary_row = {
            "Site": site_name,
            "Reference": reference_name,
            "DEM_name": dem_name,
            "Resolution": res,
            "Sensor_group": sensor_group,

            "dx_before": shifts["dx_before"],
            "dy_before": shifts["dy_before"],
            "dz_before": shifts["dz_before"],
            "horiz_before": shifts["horiz_before"],

            "dx_after": shifts["dx_after"],
            "dy_after": shifts["dy_after"],
            "dz_after": shifts["dz_after"],
            "horiz_after": shifts["horiz_after"],

            "count_weight_before": weighted_before["count"],
            "mean_weight_before": weighted_before["mean"],
            "median_weight_before": weighted_before["median"],
            "std_weight_before": weighted_before["std"],
            "nmad_weight_before": weighted_before["nmad"],
            "rmse_weight_before": weighted_before["rmse"],
            "mae_weight_before": weighted_before["mae"],

            "count_weight_after_nuth": weighted_after_nuth["count"],
            "mean_weight_after_nuth": weighted_after_nuth["mean"],
            "median_weight_after_nuth": weighted_after_nuth["median"],
            "std_weight_after_nuth": weighted_after_nuth["std"],
            "nmad_weight_after_nuth": weighted_after_nuth["nmad"],
            "rmse_weight_after_nuth": weighted_after_nuth["rmse"],
            "mae_weight_after_nuth": weighted_after_nuth["mae"],

            "count_weight_after_final": weighted_after["count"],
            "mean_weight_after_final": weighted_after["mean"],
            "median_weight_after_final": weighted_after["median"],
            "std_weight_after_final": weighted_after["std"],
            "nmad_weight_after_final": weighted_after["nmad"],
            "rmse_weight_after_final": weighted_after["rmse"],
            "mae_weight_after_final": weighted_after["mae"],

            "count_all_before": all_before["count"],
            "mean_all_before": all_before["mean"],
            "median_all_before": all_before["median"],
            "std_all_before": all_before["std"],
            "nmad_all_before": all_before["nmad"],
            "rmse_all_before": all_before["rmse"],
            "mae_all_before": all_before["mae"],

            "count_all_after_nuth": all_after_nuth["count"],
            "mean_all_after_nuth": all_after_nuth["mean"],
            "median_all_after_nuth": all_after_nuth["median"],
            "std_all_after_nuth": all_after_nuth["std"],
            "nmad_all_after_nuth": all_after_nuth["nmad"],
            "rmse_all_after_nuth": all_after_nuth["rmse"],
            "mae_all_after_nuth": all_after_nuth["mae"],

            "count_all_after_final": all_after["count"],
            "mean_all_after_final": all_after["mean"],
            "median_all_after_final": all_after["median"],
            "std_all_after_final": all_after["std"],
            "nmad_all_after_final": all_after["nmad"],
            "rmse_all_after_final": all_after["rmse"],
            "mae_all_after_final": all_after["mae"],

            "final_stage": final_stage_label,

            "n_initial_candidate_polygons": len(initial_candidate_uids),
            "n_selected_stable_polygons": len(current_stable_area),
            "n_rejected_stable_polygons_this_sensor": len(rejected_uids_this_sensor),
            "n_coreg_pixels": n_coreg_pixels,

            "auto_rmse_outlier_k": AUTO_RMSE_OUTLIER_K,
            "use_iterative_stable_selection": USE_ITERATIVE_STABLE_SELECTION,
            "stable_selection_converged": converged,
            "stopped_by_min_pixels": stopped_by_min_pixels,
            "min_coreg_pixels": min_coreg_pixels,
        }

        # Add Deramp diagnostics to summary
        for k, v in deramp_info.items():
            summary_row[k] = v

        shifts_log.append(summary_row)

    # =========================================================
    # SAVE PIXEL-LEVEL TABLE
    # =========================================================
    if len(pixel_log) > 0:
        pixel_df = pd.DataFrame(pixel_log)

        pixel_csv = os.path.join(
            output_analysis,
            f"{site_name}_coreg_pixel_dod_long.csv"
        )

        pixel_df.to_csv(pixel_csv, index=False)

        print(f"\n[INFO] Pixel-level DoD table saved: {pixel_csv}", flush=True)
        print(pixel_df.head().to_string(index=False), flush=True)

    # =========================================================
    # SAVE MAIN SUMMARY TABLE
    # =========================================================
    if len(shifts_log) > 0:
        summary_df = pd.DataFrame(shifts_log)

        summary_csv = os.path.join(
            output_analysis,
            f"{site_name}_coreg_summary_main.csv"
        )

        summary_df.to_csv(summary_csv, index=False)

        print(f"\n[INFO] Final summary table saved: {summary_csv}", flush=True)
        print(summary_df.to_string(index=False), flush=True)

    # =========================================================
    # SAVE FINAL CLEAN STABLE-AREA SHAPEFILES
    # Same location as original stable shapefile
    # =========================================================
    if len(all_rejected_stable_uids) > 0:

        clean_stable_area = stable_area[
            ~stable_area["stable_uid"].astype(int).isin(all_rejected_stable_uids)
        ].copy()

        rejected_stable_area = stable_area[
            stable_area["stable_uid"].astype(int).isin(all_rejected_stable_uids)
        ].copy()

    else:
        clean_stable_area = stable_area.copy()

        rejected_stable_area = gpd.GeoDataFrame(
            columns=stable_area.columns,
            geometry=[],
            crs=stable_area.crs,
        )

    stable_area_dir = os.path.dirname(str(Shapefiles["stable_area_shp"]))

    clean_stable_path = os.path.join(
        stable_area_dir,
        f"{site_name}_clean_stable_area_after_all_sensors_{res}.shp"
    )

    rejected_stable_path = os.path.join(
        stable_area_dir,
        f"{site_name}_rejected_stable_area_after_all_sensors_{res}.shp"
    )

    clean_stable_area.to_file(clean_stable_path)
    rejected_stable_area.to_file(rejected_stable_path)

    print(f"\n[INFO] Clean stable-area shapefile saved: {clean_stable_path}", flush=True)
    print(f"[INFO] Rejected stable-area shapefile saved: {rejected_stable_path}", flush=True)
    print(f"[INFO] Original stable polygons: {len(stable_area)}", flush=True)
    print(f"[INFO] Unique rejected polygons: {len(rejected_stable_area)}", flush=True)
    print(f"[INFO] Final clean polygons: {len(clean_stable_area)}", flush=True)

    # =========================================================
    # SAVE SENSOR-WISE REJECTION LOG
    # =========================================================
    if len(sensor_rejection_log) > 0:
        rejection_df = pd.DataFrame(sensor_rejection_log)

        rejection_csv = os.path.join(
            output_analysis,
            f"{site_name}_stable_area_rejection_log_{res}.csv"
        )

        rejection_df.to_csv(rejection_csv, index=False)

        print(f"[INFO] Sensor-wise rejection log saved: {rejection_csv}", flush=True)

        rejection_summary = (
            rejection_df
            .groupby("stable_uid")["DEM_name"]
            .apply(lambda x: ",".join(sorted(set(x))))
            .reset_index()
            .rename(columns={"DEM_name": "rejected_by"})
        )

        rejection_summary_csv = os.path.join(
            output_analysis,
            f"{site_name}_stable_area_rejection_summary_{res}.csv"
        )

        rejection_summary.to_csv(rejection_summary_csv, index=False)

        print(f"[INFO] Rejection summary saved: {rejection_summary_csv}", flush=True)